In [ ]:
from providers.openrouter import openrouter_chat
from pypdf import PdfReader
import gradio as gr
import json

In [2]:
reader = PdfReader("twin/github.pdf")
github = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        github += text

# print("PDF loaded successfully:", github)

In [3]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.
If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{github}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

# response = openrouter_chat(messages=messages)

In [4]:
def chat(message, history):
    messages = [
        {"role": "system", "content": system_prompt}
    ] + history + [
        {"role": "user", "content": message}
    ]
    return openrouter_chat(messages=messages)

# chat("Please summarize who you are", [])

In [ ]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESS

In [6]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

# record_email_tool("test@testy.com")

In [7]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

tools = [
  {"type": "function", "function": record_email_tool_json}
]

# tools

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openrouter_chat(messages=messages, tools=tools, return_completion=True)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        messages.append(message)
        tool_call = message.tool_calls[0]
        email = json.loads(tool_call.function.arguments).get("email")
        record_email_tool(email)
        messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openrouter_chat(messages=messages, tools=tools, return_completion=True)

    return response.choices[0].message.content

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Response: ChatCompletion(id='gen-1792587689-Jt2NDZpaMJzOfPZZYPjr', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Hi there! 👋 I'm Mouad Maaroufi's digital twin. Welcome to my portfolio!\n\nI'm a Full Stack Web Developer based in Rabat, Morocco, with a passion for problem-solving and first-principle thinking. I work with technologies like React.js, Vue.js, Next.js, Node.js, MongoDB, and more.\n\nWhether you're a potential client, future employer, or just curious about my work - feel free to ask me about my projects, tech stack, experience, or anything else related to my career!\n\nWhat brings you here today?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user said "hi" - this is a friendly greeting. I should respond as Mouad\'s digital twin, being professional and engaging.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user said "hi" - this is a frien

/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Response: ChatCompletion(id='gen-1792587855-yj2KgALhFagsex6GBpXQ', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-17804c8e-27c5-4428-89ee-d18f3d455f5a', function=Function(arguments='{"email":"mouadmaa@gmail.com"}', name='record_email_tool'), type='function', index=0)], reasoning='The user has provided an email address. I should use the record_email_tool to record this. Let me do that.', reasoning_details=[{'type': 'reasoning.text', 'text': 'The user has provided an email address. I should use the record_email_tool to record this. Let me do that.', 'format': 'unknown', 'index': 0}]), native_finish_reason='tool_calls')], created=1792587855, model='nvidia/nemotron-3-ultra-550b-a55b:free', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(complet

/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openrouter_chat(messages=messages, tools=tools, return_completion=True)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            print(f"Tool called to record an email: {email}")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openrouter_chat(messages=messages, tools=tools, return_completion=True)

    return response.choices[0].message.content

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: mouad@maa.com


/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called to record an email: testemail@test.io


/home/mouad/Documents/Learning/agents-course/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
